# Optimize MoveNet to TensorRT

**Run on Jetson Nano** (engine files are GPU-architecture-specific — a Nano engine
won't load on a PC and vice-versa).

**Goal**: turn `pose_raw.onnx` into a `.engine` file that runs ~5-10× faster
than vanilla PyTorch on the same hardware, and verify the speedup.

## What TensorRT actually does for you

Internally, the engine builder applies a stack of optimizations:

1. **Layer fusion** — `Conv + BN + ReLU` becomes one CUDA kernel, saving memory R/W
2. **Precision reduction** — FP32 → FP16 doubles throughput on Nano with negligible accuracy loss
3. **Kernel autotuning** — for each op, TRT benchmarks several CUDA implementations and picks the fastest for *this exact GPU*
4. **Memory reuse** — intermediate tensors share buffers when their lifetimes don't overlap
5. **Dead op elimination** — Dropout, training-only BN stats, etc. are stripped

Note: Maxwell GPU (Nano) does NOT support INT8 — so FP16 is the lowest precision
you can use here. INT8 is a Xavier/Orin feature.

## Steps

| Step | What |
|---|---|
| 0 | Environment + ONNX sanity check |
| 1 | Build the FP16 engine |
| 2 | Inspect the engine (inputs, outputs, sizes) |
| 3 | Benchmark TRT inference latency |
| 4 | (Optional) Compare against PyTorch baseline |
| 5 | (Optional) Build an FP32 engine as accuracy reference |

## Step 0: Environment + ONNX sanity check

Confirm TensorRT and PyCUDA are available, and the ONNX file is in place.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))

import tensorrt as trt
import pycuda.driver as cuda
import pycuda.autoinit  # noqa
import numpy as np

print("Python  :", sys.version.split()[0])
print("TensorRT:", trt.__version__)
print("NumPy   :", np.__version__)

# GPU info
dev = cuda.Device(0)
print("\nGPU     :", dev.name())
print("  total memory: {:.1f} GB".format(
    dev.total_memory() / (1024**3)))
print("  compute cap : {}.{}".format(*dev.compute_capability()))

# Maxwell = compute capability 5.x, supports FP16 but NOT INT8.
cc_major, _ = dev.compute_capability()
if cc_major == 5:
    print("  -> Maxwell architecture (Jetson Nano). FP16 OK, INT8 NOT supported.")
elif cc_major >= 7:
    print("  -> Volta or newer. INT8 also available.")

Python  : 3.6.9
TensorRT: 8.2.1.8
NumPy   : 1.19.5

GPU     : NVIDIA Tegra X1
  total memory: 1.9 GB
  compute cap : 5.3
  -> Maxwell architecture (Jetson Nano). FP16 OK, INT8 NOT supported.


In [2]:
# ============== Paths ==============
ONNX_PATH        = "output/pose_raw.onnx"     # input
ENGINE_FP16_PATH = "output/pose_fp16.engine"  # main output
# ENGINE_FP32_PATH = "output/pose_fp32.engine"  # optional baseline
# ===================================

if not os.path.exists(ONNX_PATH):
    raise FileNotFoundError(
        "ONNX file not found: {}\n"
        "Run pth2onnx.py on PC first, then scp it to the Nano.".format(ONNX_PATH))

size_mb = os.path.getsize(ONNX_PATH) / (1024 * 1024)
print("[OK] ONNX: {} ({:.1f} MB)".format(ONNX_PATH, size_mb))

[OK] ONNX: output/pose_raw.onnx (7.3 MB)


---
## Step 1: Build the FP16 engine

Uses `build_engine()` from `movenet/trt_builder.py`. The function:
1. Parses the ONNX with `trt.OnnxParser`
2. Configures the builder (FP16 flag + workspace size)
3. Calls `build_serialized_network()` and writes the bytes to disk

### About `workspace_gb`
TRT uses scratch memory during the *build* (kernel autotuning needs it). 1 GB is
enough for MoveNet on Nano. If you see `[ERROR] could not find any
implementation` in the build logs, raise this. If you OOM during build, lower it
or enable swap.

### About `verbose=True`
Set `verbose=True` if the build fails — it'll dump every layer's autotuning
decision. Otherwise it's noisy.

In [3]:
from trt_builder import build_engine

build_engine(
    onnx_path    = ONNX_PATH,
    engine_path  = ENGINE_FP16_PATH,
    fp16         = True,
    workspace_gb = 1,
    rebuild      = False,    # True forces a rebuild even if the file exists
    verbose      = False,
)

[INFO] ONNX  : output/pose_raw.onnx (7.3 MB)
[INFO] Engine: output/pose_fp16.engine
[INFO] Config: FP16=True, workspace=1 GB
[INFO] TensorRT 8.2.1.8
[INFO] Building engine (5-15 min on Nano)...
[OK] ONNX parsed
  Input  : input (1, 3, 192, 192)
  Output : heatmap (1, 17, 48, 48)
  Output : center (1, 1, 48, 48)
  Output : regs (1, 34, 48, 48)
  Output : offsets (1, 34, 48, 48)
[OK] FP16 enabled

[DONE] Engine saved: output/pose_fp16.engine (5.3 MB)
       Build time : 195 s


'output/pose_fp16.engine'

If the cell above failed with OOM:
```bash
# enable 4 GB swap, then re-run
sudo fallocate -l 4G /swapfile
sudo chmod 600 /swapfile
sudo mkswap /swapfile
sudo swapon /swapfile

# also free up RAM by stopping the desktop
sudo init 3
```
Then re-run the build cell.

---
## Step 2: Inspect the engine

Load the engine and print its bindings. Useful to confirm the I/O contract
matches what the downstream code expects.

In [4]:
TRT_LOGGER = trt.Logger(trt.Logger.WARNING)

with open(ENGINE_FP16_PATH, 'rb') as f:
    runtime = trt.Runtime(TRT_LOGGER)
    engine = runtime.deserialize_cuda_engine(f.read())

print("Engine: {} ({:.2f} MB)".format(
    ENGINE_FP16_PATH,
    os.path.getsize(ENGINE_FP16_PATH) / (1024 * 1024)))
print("Bindings:")
for i in range(engine.num_bindings):
    name = engine[i]
    shape = tuple(engine.get_binding_shape(i))
    dtype = trt.nptype(engine.get_binding_dtype(i))
    kind = "input " if engine.binding_is_input(i) else "output"
    print("  [{}] {} | {} {} | {}".format(
        i, kind, name, shape, np.dtype(dtype).name))

# Expected: 1 input (1,3,192,192) + 4 outputs (heatmap, center, regs, offsets)

Engine: output/pose_fp16.engine (5.26 MB)
Bindings:
  [0] input  | input (1, 3, 192, 192) | float32
  [1] output | heatmap (1, 17, 48, 48) | float32
  [2] output | center (1, 1, 48, 48) | float32
  [3] output | regs (1, 34, 48, 48) | float32
  [4] output | offsets (1, 34, 48, 48) | float32
